In [1]:
import socket

def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(('127.0.0.1', port)) == 0

if not port_open(6000):
    import subprocess, time
    subprocess.Popen(['bash', '/srv/start_mlflow.sh'])
    time.sleep(4)
    print('MLflow iniciado — esperando 4s...')
else:
    print('MLflow ya corriendo en :6000')

import mlflow
mlflow.set_tracking_uri('http://localhost:6000')
mlflow.set_experiment('latam2cloud')
print(f'MLflow OK — {mlflow.get_tracking_uri()}  |  experimento: latam2cloud')

MLflow PID: 630
MLflow iniciado — esperando 4s...
MLflow OK — http://localhost:6000  |  experimento: latam2cloud


In [2]:

import os, sys, time, gc, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)

warnings.filterwarnings('ignore')

SAVE_DIR   = '/srv/work/models'
IN_FEAT_V5 = '/srv/work/te_features_v5.parquet'
IN_FEAT_V6 = '/srv/work/te_features_v6.parquet'

ROUTE_KEYS = ['source_agent', 'cloud_provider', 'region']
ID_COLS    = ROUTE_KEYS + ['window']
SPLIT_COL  = 'split'
TARGET_COL = 'label_ventana'
HORIZON    = 2

if '/srv/python_packages' not in sys.path:
    sys.path.insert(0, '/srv/python_packages')

T1_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.01), 2)

LGB_PARAMS_V14 = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'num_leaves': 63,
    'learning_rate': 0.03,
    'n_estimators': 800,
    'max_depth': -1,
    'min_child_samples': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_jobs': 8,
    'random_state': 42,
    'verbosity': -1,
}

SMOTE_ALTA_RATIO = 0.15
N_SUBSAMPLE      = 1_500_000

QWK_FLOOR     = 0.65
RECALTA_FLOOR = 0.55

try:
    from imblearn.over_sampling import SMOTE
    print('imbalanced-learn disponible OK')
except ImportError:
    import subprocess
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'imbalanced-learn', '--target', '/srv/python_packages'
    ])
    from imblearn.over_sampling import SMOTE
    print('imbalanced-learn instalado OK')

import lightgbm as lgb
print(f'LightGBM {lgb.__version__}')
print(f'T1 thresholds: {len(T1_THRESHOLDS)} [{T1_THRESHOLDS[0]}–{T1_THRESHOLDS[-1]}]')
print(f'SMOTE Alta target: {SMOTE_ALTA_RATIO*100:.0f}%  |  N_SUBSAMPLE: {N_SUBSAMPLE:,}')

imbalanced-learn disponible OK
LightGBM 4.7.0
T1 thresholds: 46 [0.05–0.5]
SMOTE Alta target: 15%  |  N_SUBSAMPLE: 1,500,000


In [3]:

def evaluar(y_true, y_pred_proba, nombre,
            floor_qwk=QWK_FLOOR, floor_rec=RECALTA_FLOOR,
            t_threshold=None):
    if t_threshold is not None:
        y_pred = np.argmax(y_pred_proba, axis=1).copy()
        y_pred[y_pred_proba[:, 3] > t_threshold] = 3
    else:
        y_pred = np.argmax(y_pred_proba, axis=1)

    qwk      = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    cm       = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    rec_alta = cm[3,3] / cm[3,:].sum() if cm[3,:].sum() > 0 else 0.0
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bal_acc  = balanced_accuracy_score(y_true, y_pred)
    ok_qwk   = qwk >= floor_qwk
    ok_rec   = rec_alta >= floor_rec
    cand     = 'SI' if (ok_qwk and ok_rec) else 'NO'
    thr_str  = f' [T={t_threshold}]' if t_threshold else ''

    print('-'*80)
    print(f'  {nombre}{thr_str}')
    print('-'*80)
    print(f'  QWK      = {qwk:.4f}  {"OK" if ok_qwk else "X"}  (floor>={floor_qwk})')
    print(f'  RecAlta  = {rec_alta:.4f}  {"OK" if ok_rec else "X"}  (floor>={floor_rec})')
    print(f'  MacroF1  = {macro_f1:.4f}')
    print(f'  BalAcc   = {bal_acc:.4f}')
    print(f'  CANDIDATO: {cand}')
    for k, nm in enumerate(['Normal','Baja','Media','Alta']):
        print(f'    {nm:<8}: ' + '  '.join(f'{cm[k,j]:>9,}' for j in range(4)))
    return qwk, rec_alta, macro_f1, bal_acc

print('evaluar() definida.')

evaluar() definida.


In [4]:

print('R5: Cargando columnas minimas de te_features_v5.parquet...')
keep_v5 = ROUTE_KEYS + [TARGET_COL, SPLIT_COL]

df_v5_all = pq.read_table(IN_FEAT_V5, columns=keep_v5).to_pandas()
df_v5_all['y'] = df_v5_all.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_v5_all = df_v5_all.dropna(subset=['y']).copy()
df_v5_all['y'] = df_v5_all['y'].astype(int)

val_mask = df_v5_all[SPLIT_COL] == 'val'
y_val_v9 = df_v5_all.loc[val_mask, 'y'].values
del df_v5_all; gc.collect()

print(f'  y_val_v9: {y_val_v9.shape}')
n_alta_v9 = (y_val_v9 == 3).sum()
print(f'  Alta en val_v9: {n_alta_v9:,} ({100*n_alta_v9/len(y_val_v9):.1f}%)')
print('Celda 3 OK')

R5: Cargando columnas minimas de te_features_v5.parquet...
  y_val_v9: (644690,)
  Alta en val_v9: 20,085 (3.1%)
Celda 3 OK


In [5]:

print('R5: Cargando ens_v9_val_proba.npy...')
proba_ens_v9 = np.load(f'{SAVE_DIR}/ens_v9_val_proba.npy')
print(f'  ens_v9_val_proba: {proba_ens_v9.shape}')
print(f'  y_val_v9        : {y_val_v9.shape}')

if proba_ens_v9.shape[0] != y_val_v9.shape[0]:
    raise ValueError(
        f'Dimensiones incompatibles: proba={proba_ens_v9.shape[0]}, '
        f'y_val={y_val_v9.shape[0]}'
    )

y_pred_base = np.argmax(proba_ens_v9, axis=1)
qwk_base  = cohen_kappa_score(y_val_v9, y_pred_base, weights='quadratic')
cm_base   = confusion_matrix(y_val_v9, y_pred_base, labels=[0,1,2,3])
rec_base  = cm_base[3,3] / cm_base[3,:].sum() if cm_base[3,:].sum() > 0 else 0.0
print(f'\n  Baseline v9 argmax: QWK={qwk_base:.4f}, RecAlta={rec_base:.4f}')
print(f'  (Referencia LOG-25: QWK=0.6529, RecAlta=0.6509)')
print('Celda 4 OK')

R5: Cargando ens_v9_val_proba.npy...
  ens_v9_val_proba: (644690, 4)
  y_val_v9        : (644690,)

  Baseline v9 argmax: QWK=0.6529, RecAlta=0.6509
  (Referencia LOG-25: QWK=0.6529, RecAlta=0.6509)
Celda 4 OK


In [6]:

print('R5: Escaneo T1 sobre ens_v9_val_proba...')
print(f'  {"T":>6}  {"QWK":>8}  {"RecAlta":>8}  Cand')
print('  ' + '-'*38)

t1_results = []
for T in T1_THRESHOLDS:
    y_pred = np.argmax(proba_ens_v9, axis=1).copy()
    y_pred[proba_ens_v9[:, 3] > T] = 3
    qwk      = cohen_kappa_score(y_val_v9, y_pred, weights='quadratic')
    cm       = confusion_matrix(y_val_v9, y_pred, labels=[0,1,2,3])
    rec_alta = cm[3,3] / cm[3,:].sum() if cm[3,:].sum() > 0 else 0.0
    is_cand  = (qwk >= QWK_FLOOR) and (rec_alta >= RECALTA_FLOOR)
    t1_results.append((T, qwk, rec_alta, is_cand))
    mark = 'OK' if is_cand else ''
    print(f'  {T:>6.2f}  {qwk:>8.4f}  {rec_alta:>8.4f}   {mark}')

candidates_r5 = [(T, qwk, rec) for T, qwk, rec, c in t1_results if c]
print(f'\nR5: {len(candidates_r5)} thresholds candidatos encontrados')

R5: Escaneo T1 sobre ens_v9_val_proba...
       T       QWK   RecAlta  Cand
  --------------------------------------
    0.05    0.4034    0.9356   
    0.06    0.4341    0.9240   
    0.07    0.4593    0.9148   
    0.08    0.4801    0.9054   
    0.09    0.4983    0.8960   
    0.10    0.5141    0.8865   
    0.11    0.5285    0.8783   
    0.12    0.5412    0.8701   
    0.13    0.5525    0.8605   
    0.14    0.5628    0.8526   
    0.15    0.5720    0.8439   
    0.16    0.5802    0.8360   
    0.17    0.5876    0.8281   
    0.18    0.5944    0.8190   
    0.19    0.6004    0.8108   
    0.20    0.6056    0.8020   
    0.21    0.6119    0.7943   
    0.22    0.6161    0.7854   
    0.23    0.6204    0.7762   
    0.24    0.6241    0.7675   
    0.25    0.6271    0.7590   
    0.26    0.6303    0.7510   
    0.27    0.6330    0.7412   
    0.28    0.6356    0.7328   
    0.29    0.6382    0.7248   
    0.30    0.6405    0.7166   
    0.31    0.6425    0.7094   
    0.32    0.6441 

In [7]:

if candidates_r5:
    T_best_r5, _, _ = max(candidates_r5, key=lambda x: x[2])
    print(f'R5 MEJOR THRESHOLD (max RecAlta): T={T_best_r5}')
    qwk_r5, rec_r5, f1_r5, ba_r5 = evaluar(
        y_val_v9, proba_ens_v9,
        'R5: Ens v9 + T1 threshold (max RecAlta)',
        t_threshold=T_best_r5
    )
    print(f'\n  Delta QWK     = {qwk_r5 - qwk_base:+.4f}')
    print(f'  Delta RecAlta = {rec_r5 - rec_base:+.4f}')
    T_best_r5_qwk, qwk_r5_max, rec_r5_qwk = max(candidates_r5, key=lambda x: x[1])
    if T_best_r5_qwk != T_best_r5:
        print(f'  (Candidato max QWK: T={T_best_r5_qwk}, '
              f'QWK={qwk_r5_max:.4f}, RecAlta={rec_r5_qwk:.4f})')
else:
    T_best_r5 = None; qwk_r5 = None; rec_r5 = None; f1_r5 = None; ba_r5 = None
    print('R5: ningun threshold genera candidato.')
    print(f'  Mejor QWK:    {max(t1_results, key=lambda x: x[1])}')
    print(f'  Mejor RecAlta: {max(t1_results, key=lambda x: x[2])}')

print('Celda 6 OK')

R5 MEJOR THRESHOLD (max RecAlta): T=0.38
--------------------------------------------------------------------------------
  R5: Ens v9 + T1 threshold (max RecAlta) [T=0.38]
--------------------------------------------------------------------------------
  QWK      = 0.6504  OK  (floor>=0.65)
  RecAlta  = 0.6697  OK  (floor>=0.55)
  MacroF1  = 0.6618
  BalAcc   = 0.6688
  CANDIDATO: SI
    Normal  :   524,090     15,407      4,164      7,530
    Baja    :    17,110     38,399      1,805      1,331
    Media   :     4,882      1,895      5,908      2,084
    Alta    :     4,763        591      1,280     13,451

  Delta QWK     = -0.0024
  Delta RecAlta = +0.0188
  (Candidato max QWK: T=0.49, QWK=0.6529, RecAlta=0.6509)
Celda 6 OK


In [8]:

print('R1: Cargando te_features_v6.parquet (split=train)...')

schema    = pq.read_schema(IN_FEAT_V6)
all_cols  = [f.name for f in schema]
exclude   = set(ID_COLS + [TARGET_COL, SPLIT_COL])
FEAT_COLS = [c for c in all_cols if c not in exclude]
keep_cols = ID_COLS + [TARGET_COL, SPLIT_COL] + FEAT_COLS

df_train = pq.read_table(
    IN_FEAT_V6, columns=keep_cols,
    filters=[('split', '=', 'train')]
).to_pandas()

df_train['y'] = df_train.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_train = df_train.dropna(subset=['y']).copy()
df_train['y'] = df_train['y'].astype(int)

X_full = df_train[FEAT_COLS].values.astype(np.float32)
y_full = df_train['y'].values
n_train_full = len(y_full)
del df_train; gc.collect()

counts_full = np.bincount(y_full, minlength=4)
n_alta_full = counts_full[3]
print(f'  Train completo: {X_full.shape}  (~{X_full.nbytes/1e9:.2f} GB)')
for k, nm in enumerate(['Normal','Baja','Media','Alta']):
    print(f'    {nm:<8}: {counts_full[k]:>9,}  ({100*counts_full[k]/n_train_full:.1f}%)')

rng          = np.random.default_rng(42)
idx_alta     = np.where(y_full == 3)[0]
idx_no_alta  = np.where(y_full != 3)[0]
n_sub_rest   = min(N_SUBSAMPLE, len(idx_no_alta))
idx_sub_rest = rng.choice(idx_no_alta, n_sub_rest, replace=False)
idx_sub      = np.concatenate([idx_alta, idx_sub_rest])
rng.shuffle(idx_sub)

X_train_v6 = X_full[idx_sub]
y_train_v6 = y_full[idx_sub]
n_train    = len(y_train_v6)
del X_full, y_full, idx_sub, idx_no_alta, idx_sub_rest; gc.collect()

ratio_idx = [i for i, c in enumerate(FEAT_COLS) if 'ratio' in c]
other_idx  = [i for i, c in enumerate(FEAT_COLS) if 'ratio' not in c]
for idx in ratio_idx:
    mask = np.isnan(X_train_v6[:, idx])
    if mask.any(): X_train_v6[mask, idx] = 1.0
for idx in other_idx:
    mask = np.isnan(X_train_v6[:, idx])
    if mask.any(): X_train_v6[mask, idx] = 0.0
n_nan_after = int(np.isnan(X_train_v6).sum())
print(f'  NaN imputados (ratio→1.0, resto→0.0) — NaN restantes: {n_nan_after}')

counts_train = np.bincount(y_train_v6, minlength=4)
n_alta_train = counts_train[3]
print(f'\n  Post-subsampleo: {X_train_v6.shape}  (~{X_train_v6.nbytes/1e9:.2f} GB)')
for k, nm in enumerate(['Normal','Baja','Media','Alta']):
    print(f'    {nm:<8}: {counts_train[k]:>9,}  ({100*counts_train[k]/n_train:.1f}%)')

n_no_alta   = n_train - n_alta_train
target_alta = int(SMOTE_ALTA_RATIO * n_no_alta / (1 - SMOTE_ALTA_RATIO))
target_alta = max(target_alta, n_alta_train)
print(f'\n  Alta actual: {n_alta_train:,} ({100*n_alta_train/n_train:.1f}%)')
print(f'  Alta objetivo post-SMOTE: {target_alta:,} ({SMOTE_ALTA_RATIO*100:.0f}%)')
print(f'  Nuevas muestras sinteticas: {target_alta - n_alta_train:,}')
print('Celda 7 OK')

R1: Cargando te_features_v6.parquet (split=train)...
  Train completo: (5347524, 103)  (~2.20 GB)
    Normal  : 4,619,737  (86.4%)
    Baja    :   509,101  (9.5%)
    Media   :   112,159  (2.1%)
    Alta    :   106,527  (2.0%)
  NaN imputados (ratio→1.0, resto→0.0) — NaN restantes: 0

  Post-subsampleo: (1606527, 103)  (~0.66 GB)
    Normal  : 1,322,633  (82.3%)
    Baja    :   145,311  (9.0%)
    Media   :    32,056  (2.0%)
    Alta    :   106,527  (6.6%)

  Alta actual: 106,527 (6.6%)
  Alta objetivo post-SMOTE: 264,705 (15%)
  Nuevas muestras sinteticas: 158,178
Celda 7 OK


In [9]:

print('R1: Aplicando SMOTE (k_neighbors=5, solo clase Alta)...')
print(f'  X_train_v6: {X_train_v6.shape}  ({X_train_v6.nbytes/1e9:.2f} GB)')
t_smote = time.time()

smote = SMOTE(
    sampling_strategy={3: target_alta},
    k_neighbors=5,
    random_state=42,
)

X_sm, y_sm = smote.fit_resample(X_train_v6, y_train_v6)
del X_train_v6, y_train_v6; gc.collect()

t_elapsed = time.time() - t_smote
counts_sm = np.bincount(y_sm, minlength=4)
n_sm      = len(y_sm)

print(f'  SMOTE completado en {t_elapsed/60:.1f} min')
print(f'  X_sm: {X_sm.shape}  (~{X_sm.nbytes/1e9:.2f} GB)')
for k, nm in enumerate(['Normal','Baja','Media','Alta']):
    print(f'    {nm:<8}: {counts_sm[k]:>9,}  ({100*counts_sm[k]/n_sm:.1f}%)')
print('Celda 8 OK')

R1: Aplicando SMOTE (k_neighbors=5, solo clase Alta)...
  X_train_v6: (1606527, 103)  (0.66 GB)
  SMOTE completado en 0.4 min
  X_sm: (1764705, 103)  (~0.73 GB)
    Normal  : 1,322,633  (74.9%)
    Baja    :   145,311  (8.2%)
    Media   :    32,056  (1.8%)
    Alta    :   264,705  (15.0%)
Celda 8 OK


In [10]:

print('R1: Cargando val v6 para early stopping...')
df_val = pq.read_table(
    IN_FEAT_V6, columns=keep_cols,
    filters=[('split', '=', 'val')]
).to_pandas()
df_val['y'] = df_val.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
X_val_v6  = df_val[FEAT_COLS].values.astype(np.float32)
y_val_raw = df_val['y'].fillna(-1).astype(int).values
mask_valid = y_val_raw >= 0
X_val_v6   = X_val_v6[mask_valid]
y_val_v6   = y_val_raw[mask_valid]
del df_val; gc.collect()

for idx in ratio_idx:
    mask = np.isnan(X_val_v6[:, idx])
    if mask.any(): X_val_v6[mask, idx] = 1.0
for idx in other_idx:
    mask = np.isnan(X_val_v6[:, idx])
    if mask.any(): X_val_v6[mask, idx] = 0.0
print(f'  X_val_v6: {X_val_v6.shape}  NaN restantes: {int(np.isnan(X_val_v6).sum())}')

print(f'\nR1: Entrenando LGB v14 ({X_sm.shape[0]:,} muestras SMOTE)...')
t_start = time.time()
lgb_v14 = lgb.LGBMClassifier(**LGB_PARAMS_V14)
lgb_v14.fit(
    X_sm, y_sm,
    eval_set=[(X_val_v6, y_val_v6)],
    callbacks=[
        lgb.log_evaluation(100),
        lgb.early_stopping(stopping_rounds=50, verbose=False),
    ],
)
t_elapsed = time.time() - t_start
print(f'  Entrenado en {t_elapsed/60:.1f} min')

del X_sm, y_sm; gc.collect()
print('  Datos SMOTE liberados')

proba_lgb_v14 = lgb_v14.predict_proba(X_val_v6)
np.save(f'{SAVE_DIR}/lgb_v14_val_proba.npy', proba_lgb_v14)
lgb_v14.booster_.save_model(f'{SAVE_DIR}/lgb_v14_model.txt')
print(f'  Guardado: lgb_v14_val_proba.npy + lgb_v14_model.txt')

qwk_lgb14, rec_lgb14, f1_lgb14, ba_lgb14 = evaluar(
    y_val_v6, proba_lgb_v14,
    'R1: LGB v14 (SMOTE subsampleo, v13 params)'
)
print('Celda 9 OK')

R1: Cargando val v6 para early stopping...
  X_val_v6: (1138777, 103)  NaN restantes: 0

R1: Entrenando LGB v14 (1,764,705 muestras SMOTE)...
[100]	valid_0's multi_logloss: 0.198458
[200]	valid_0's multi_logloss: 0.18837
[300]	valid_0's multi_logloss: 0.185662
[400]	valid_0's multi_logloss: 0.184755
[500]	valid_0's multi_logloss: 0.184363
  Entrenado en 6.7 min
  Datos SMOTE liberados
  Guardado: lgb_v14_val_proba.npy + lgb_v14_model.txt
--------------------------------------------------------------------------------
  R1: LGB v14 (SMOTE subsampleo, v13 params)
--------------------------------------------------------------------------------
  QWK      = 0.6097  X  (floor>=0.65)
  RecAlta  = 0.5659  OK  (floor>=0.55)
  MacroF1  = 0.6183
  BalAcc   = 0.5958
  CANDIDATO: NO
    Normal  : 1,007,839      6,458        397      9,456
    Baja    :    21,796     45,840        711      2,171
    Media   :    10,160      2,687      4,145      5,614
    Alta    :     8,614        434        287  

In [11]:

print('R1: Cargando RF v10 checkpoint...')
proba_rf_v10 = np.load(f'{SAVE_DIR}/rf_v10_val_proba.npy')
print(f'  rf_v10_val_proba:  {proba_rf_v10.shape}')
print(f'  lgb_v14_val_proba: {proba_lgb_v14.shape}')

n_min = min(proba_rf_v10.shape[0], proba_lgb_v14.shape[0])
if proba_rf_v10.shape[0] != proba_lgb_v14.shape[0]:
    print(f'  Shapes distintos — truncando a {n_min} filas')
proba_rf_v10_use  = proba_rf_v10[:n_min]
proba_lgb_v14_use = proba_lgb_v14[:n_min]
y_val_v6_use      = y_val_v6[:n_min]

best_r1 = None
print(f'\nGrid search LGB_v14 x RF_v10 (paso 0.05):')
print(f'  {"w_RF":>6}  {"w_LGB":>6}  {"QWK":>8}  {"RecAlta":>8}  Cand')
print('  ' + '-'*50)
for w_rf in np.arange(0.0, 1.01, 0.05):
    w_lgb = round(1.0 - w_rf, 2)
    proba_ens = w_rf * proba_rf_v10_use + w_lgb * proba_lgb_v14_use
    y_pred = np.argmax(proba_ens, axis=1)
    qwk    = cohen_kappa_score(y_val_v6_use, y_pred, weights='quadratic')
    cm     = confusion_matrix(y_val_v6_use, y_pred, labels=[0,1,2,3])
    rec    = cm[3,3] / cm[3,:].sum() if cm[3,:].sum() > 0 else 0.0
    is_cand = (qwk >= QWK_FLOOR) and (rec >= RECALTA_FLOOR)
    mark = 'OK' if is_cand else ''
    print(f'  {w_rf:>6.2f}  {w_lgb:>6.2f}  {qwk:>8.4f}  {rec:>8.4f}   {mark}')
    if is_cand:
        if best_r1 is None or qwk > best_r1[2]:
            best_r1 = (w_rf, w_lgb, qwk, rec, proba_ens.copy())

if best_r1:
    w_rf_r1, w_lgb_r1, qwk_r1, rec_r1, proba_ens_r1 = best_r1
    np.save(f'{SAVE_DIR}/ens_v14_val_proba.npy', proba_ens_r1)
    print(f'\nR1 CANDIDATO: RF x{w_rf_r1}+LGB x{w_lgb_r1} QWK={qwk_r1:.4f} RecAlta={rec_r1:.4f}')
    qwk_r1, rec_r1, f1_r1, ba_r1 = evaluar(
        y_val_v6_use, proba_ens_r1,
        f'R1: Ensemble v14 (RF x{w_rf_r1}+LGB x{w_lgb_r1})'
    )
else:
    qwk_r1 = None; rec_r1 = None; w_rf_r1 = None; w_lgb_r1 = None
    best_qwk_val = -1; best_by_qwk = None
    for w_rf in np.arange(0.0, 1.01, 0.05):
        w_lgb = round(1.0 - w_rf, 2)
        proba_ens = w_rf * proba_rf_v10_use + w_lgb * proba_lgb_v14_use
        y_pred = np.argmax(proba_ens, axis=1)
        qwk    = cohen_kappa_score(y_val_v6_use, y_pred, weights='quadratic')
        if qwk > best_qwk_val:
            best_qwk_val = qwk
            cm  = confusion_matrix(y_val_v6_use, y_pred, labels=[0,1,2,3])
            rec = cm[3,3] / cm[3,:].sum() if cm[3,:].sum() > 0 else 0.0
            best_by_qwk = (w_rf, w_lgb, qwk, rec, proba_ens.copy())
    w_rf_r1b, w_lgb_r1b, qwk_r1_best, rec_r1_best, proba_ens_r1_best = best_by_qwk
    np.save(f'{SAVE_DIR}/ens_v14_val_proba.npy', proba_ens_r1_best)
    print(f'\nR1: sin candidato — mejor: RF x{w_rf_r1b}+LGB x{w_lgb_r1b}'
          f' QWK={qwk_r1_best:.4f} RecAlta={rec_r1_best:.4f}')
    evaluar(y_val_v6_use, proba_ens_r1_best,
            f'R1: Ens v14 mejor (RF x{w_rf_r1b}+LGB x{w_lgb_r1b})')
    qwk_r1 = qwk_r1_best; rec_r1 = rec_r1_best

print('Celda 10 OK')

R1: Cargando RF v10 checkpoint...
  rf_v10_val_proba:  (1138777, 4)
  lgb_v14_val_proba: (1138777, 4)

Grid search LGB_v14 x RF_v10 (paso 0.05):
    w_RF   w_LGB       QWK   RecAlta  Cand
  --------------------------------------------------
    0.00    1.00    0.6097    0.5659   
    0.05    0.95    0.6106    0.5642   
    0.10    0.90    0.6113    0.5622   
    0.15    0.85    0.6126    0.5611   
    0.20    0.80    0.6138    0.5598   
    0.25    0.75    0.6146    0.5574   
    0.30    0.70    0.6158    0.5564   
    0.35    0.65    0.6165    0.5538   
    0.40    0.60    0.6174    0.5517   
    0.45    0.55    0.6185    0.5495   
    0.50    0.50    0.6189    0.5459   
    0.55    0.45    0.6196    0.5420   
    0.60    0.40    0.6203    0.5385   
    0.65    0.35    0.6209    0.5347   
    0.70    0.30    0.6215    0.5310   
    0.75    0.25    0.6220    0.5278   
    0.80    0.20    0.6225    0.5249   
    0.85    0.15    0.6224    0.5192   
    0.90    0.10    0.6227    0.5162   

In [12]:

print('='*85)
print('  TABLA COMPARATIVA v9 → v13 → v14 (R5+R1) — val')
print('='*85)
print(f'  {"Modelo":<52}  {"QWK":>6}  {"RecAlta":>8}  Cand')
print('  ' + '-'*78)

historico = [
    ('v9  Ens D38 (v5, 83d)  BASELINE CANDIDATO', 0.6529, 0.6509, True),
    ('v10 Ens opt (v6, 162d)',                     0.6227, 0.5237, False),
    ('v11 Ens opt (v6/P1, 83d)',                   0.6231, 0.5254, False),
    ('v12 XGB Ens opt (v6/P2)',                    0.5410, 0.7048, False),
    ('v13 Ens T2 (v6, T1+T2)',                     0.6224, 0.5092, False),
]
for nm, q, r, c in historico:
    print(f'  {nm:<52}  {q:>6.4f}  {r:>8.4f}  {"OK" if c else "X"}')

print('  ' + '-'*78)

r5_cand = (T_best_r5 is not None) and (qwk_r5 is not None) and \
          (qwk_r5 >= QWK_FLOOR) and (rec_r5 >= RECALTA_FLOOR)
if T_best_r5 is not None and qwk_r5 is not None:
    print(f'  {f"v14 R5: Ens v9 + T1 (T={T_best_r5})":<52}  {qwk_r5:>6.4f}  {rec_r5:>8.4f}  {"OK" if r5_cand else "X"}')
else:
    r5_cand = False
    print(f'  {"v14 R5: sin mejora sobre baseline":<52}  {qwk_base:>6.4f}  {rec_base:>8.4f}  X')

r1_cand = (qwk_r1 is not None) and (qwk_r1 >= QWK_FLOOR) and (rec_r1 >= RECALTA_FLOOR)
if qwk_r1 is not None:
    w_str = f'RF x{w_rf_r1}+LGB x{w_lgb_r1}' if w_rf_r1 else '(mejor)'
    print(f'  {f"v14 R1: SMOTE Ens ({w_str})":<52}  {qwk_r1:>6.4f}  {rec_r1:>8.4f}  {"OK" if r1_cand else "X"}')

print('  ' + '='*78)
print(f'  Umbral: QWK>={QWK_FLOOR} AND RecAlta>={RECALTA_FLOOR}')
print()

any_new_cand = r5_cand or r1_cand
if any_new_cand:
    if r5_cand and r1_cand:
        best_new = 'R5' if qwk_r5 >= qwk_r1 else 'R1'
        print(f'  CANDIDATOS: R5 y R1 — mejor: {best_new}')
    elif r5_cand:
        print(f'  CANDIDATO R5: Ens v9 + T1 T={T_best_r5}  QWK={qwk_r5:.4f}  RecAlta={rec_r5:.4f}')
    else:
        print(f'  CANDIDATO R1: SMOTE Ens v14  QWK={qwk_r1:.4f}  RecAlta={rec_r1:.4f}')
    print('  → Proceder con 06 evaluacion test set v7.ipynb')
else:
    print(f'  Sin candidatos nuevos en v14 (R5 y R1)')
    print(f'  CANDIDATO OE2 DEFINITIVO: v9 Ensemble D38 (QWK=0.6529 RecAlta=0.6509)')
    print(f'  → Proceder con 06 evaluacion test set v7.ipynb (ONE-SHOT IRREVERSIBLE)')

print('Celda 11 OK')

  TABLA COMPARATIVA v9 → v13 → v14 (R5+R1) — val
  Modelo                                                   QWK   RecAlta  Cand
  ------------------------------------------------------------------------------
  v9  Ens D38 (v5, 83d)  BASELINE CANDIDATO             0.6529    0.6509  OK
  v10 Ens opt (v6, 162d)                                0.6227    0.5237  X
  v11 Ens opt (v6/P1, 83d)                              0.6231    0.5254  X
  v12 XGB Ens opt (v6/P2)                               0.5410    0.7048  X
  v13 Ens T2 (v6, T1+T2)                                0.6224    0.5092  X
  ------------------------------------------------------------------------------
  v14 R5: Ens v9 + T1 (T=0.38)                          0.6504    0.6697  OK
  v14 R1: SMOTE Ens ((mejor))                           0.6227    0.5162  X
  Umbral: QWK>=0.65 AND RecAlta>=0.55

  CANDIDATO R5: Ens v9 + T1 T=0.38  QWK=0.6504  RecAlta=0.6697
  → Proceder con 06 evaluacion test set v7.ipynb
Celda 11 OK


In [13]:

import mlflow

with mlflow.start_run(run_name='r5_t1_ens_v9') as run:
    mlflow.log_params({
        'strategy': 'R5_threshold_tuning',
        'base_model': 'ens_v9_D38',
        'dataset': 'v5_83d',
        't1_threshold': T_best_r5 if T_best_r5 else 'NA',
        'candidato': bool(r5_cand),
    })
    mlflow.log_metrics({'qwk_val': qwk_r5 or qwk_base,
                        'rec_alta_val': rec_r5 or rec_base})
    print(f'MLflow r5_t1_ens_v9 — {run.info.run_id}')

with mlflow.start_run(run_name='r1_lgb_v14_smote') as run:
    mlflow.log_params({
        'strategy': 'R1_SMOTE',
        'smote_alta_ratio': SMOTE_ALTA_RATIO,
        'n_subsample_no_alta': N_SUBSAMPLE,
        'dataset': 'v6_162d',
        **LGB_PARAMS_V14
    })
    mlflow.log_metrics({'qwk_val': qwk_lgb14, 'rec_alta_val': rec_lgb14,
                        'macro_f1_val': f1_lgb14, 'bal_acc_val': ba_lgb14})
    mlflow.log_artifact(f'{SAVE_DIR}/lgb_v14_model.txt')
    print(f'MLflow r1_lgb_v14_smote — {run.info.run_id}')

with mlflow.start_run(run_name='r1_ens_v14') as run:
    mlflow.log_params({
        'strategy': 'R1_SMOTE_ensemble',
        'w_rf': w_rf_r1 if w_rf_r1 else 'NA',
        'w_lgb': w_lgb_r1 if w_lgb_r1 else 'NA',
        'rf_source': 'rf_v10_ckpt',
        'lgb_source': 'lgb_v14_smote',
        'candidato': bool(r1_cand),
    })
    mlflow.log_metrics({'qwk_val': qwk_r1 or 0.0, 'rec_alta_val': rec_r1 or 0.0})
    print(f'MLflow r1_ens_v14 — {run.info.run_id}')

print()
if not any_new_cand:
    print('CANDIDATO DEFINITIVO OE2: v9 Ensemble D38')
    print('Proximo: 06 evaluacion test set v7.ipynb (ONE-SHOT IRREVERSIBLE)')
else:
    print('Candidato nuevo encontrado. Revisar tabla antes del one-shot.')
print('v14 completo.')

MLflow r5_t1_ens_v9 — 152ec3c0f2a448239319e66040ef7845
🏃 View run r5_t1_ens_v9 at: http://localhost:6000/#/experiments/681062908553326840/runs/152ec3c0f2a448239319e66040ef7845
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
MLflow r1_lgb_v14_smote — e913029fa6f24eb6ac6e49b01a697297
🏃 View run r1_lgb_v14_smote at: http://localhost:6000/#/experiments/681062908553326840/runs/e913029fa6f24eb6ac6e49b01a697297
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
MLflow r1_ens_v14 — 13b510fe9c654fbd94012a6a1ea29b8d
🏃 View run r1_ens_v14 at: http://localhost:6000/#/experiments/681062908553326840/runs/13b510fe9c654fbd94012a6a1ea29b8d
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840

Candidato nuevo encontrado. Revisar tabla antes del one-shot.
v14 completo.
